In [ ]:
pip install pandas pyarrow

In [ ]:
pip install endee

In [ ]:
pip show endee

In [ ]:
base_path = "/home/debian/latest_VDB/VectorDBBench/vectordataset"
dataset_folder = "cohere/cohere_medium_1m"

In [ ]:
#For checking parquet file contents
import pyarrow.parquet as pq
import os

file_name = "shuffle_train.parquet"  #input

# Build full path
file_path = os.path.join(base_path, dataset_folder, file_name)


parquet_file = pq.ParquetFile(file_path)

# Read only first batch of rows
first_batch = next(parquet_file.iter_batches(batch_size=5))
preview = first_batch.to_pandas()

for col in preview.columns:
    if preview[col].dtype == object and isinstance(preview[col].iloc[0], list):
        preview[col] = preview[col].apply(lambda x: x[:5] if x is not None else x)

print(preview)

In [ ]:
from endee import Endee
client = Endee(token="localtest")
client.set_base_url("http://148.113.54.173:8080/api/v1")

In [ ]:
#For checking indexes
for obj in client.list_indexes().get('indexes'):
    print(obj['name'])
    print(obj['total_elements'])
    print('\t')

In [ ]:
#give the index name
index_name = "test_1M_vaib_reupsertcheck_new1"
index = client.get_index(index_name)

In [ ]:
index.describe()

In [ ]:
#For querying (int filter)- gte filter
import pyarrow.parquet as pq
import os

#inputs
int_rate_query = "99p"   # change to 1p, 50p, 80p, 99p
top_k = 30

# Mode 1: Single query id
# Mode 2: Find all query ids with recall < 1
mode = 2
query_id = 457    # only used in mode 1

# Map int_rate_query to filter range
range_map = {
    "1p":  [10000, 1000000],
    "50p": [500000, 1000000],
    "80p": [800000, 1000000],
    "99p": [990000, 1000000],
}
filter_range = range_map[int_rate_query]

# Build full paths
dataset_path = os.path.join(base_path, dataset_folder)
test_parquet_path = os.path.join(dataset_path, "test.parquet")
gt_parquet_path = os.path.join(dataset_path, f"neighbors_int_{int_rate_query}.parquet")

# Read parquet files
test_df = pq.ParquetFile(test_parquet_path).read().to_pandas()
gt_df = pq.ParquetFile(gt_parquet_path).read().to_pandas()

def run_query(query_id):
    query_row = test_df[test_df["id"] == query_id]
    if query_row.empty:
        print(f"Query ID {query_id} not found in test.parquet")
        return None

    query_vector = query_row["emb"].values[0]

    gt_row = gt_df[gt_df["id"] == query_id]
    if gt_row.empty:
        print(f"Query ID {query_id} not found in ground truth file")
        return None

    ground_truth = gt_row["neighbors_id"].values[0][:top_k]

    results = index.query(
        vector=query_vector,
        top_k=top_k,
        filter=[{"id": {"$gte": filter_range[0]}}],
        include_vectors=False,
    )
    returned_ids = [int(r["id"]) for r in results]

    intersection = len(set(returned_ids) & set(ground_truth))
    recall = intersection / len(ground_truth)

    return returned_ids, ground_truth, recall


if mode == 1:
    result = run_query(query_id)
    if result:
        returned_ids, ground_truth, recall = result
        print(f"Query ID: {query_id}")
        print(f"Returned IDs: {','.join(map(str, returned_ids))}")
        print(f"Ground Truth: {','.join(map(str, ground_truth))}")
        print(f"Recall: {recall:.4f}")

elif mode == 2:
    print(f"Finding all query IDs with recall < 1.0 ...\n")
    low_recall_ids = []
    all_recalls = []

    for qid in test_df["id"].values:
        result = run_query(qid)
        if result:
            returned_ids, ground_truth, recall = result
            all_recalls.append(recall)
            if recall < 1.0:
                low_recall_ids.append(qid)
                print(f"Query ID: {qid}")
                print(f"Returned IDs: {','.join(map(str, returned_ids))}")
                print(f"Ground Truth: {','.join(map(str, ground_truth))}")
                print(f"Recall: {recall:.4f}\n")

    print(f"Total query IDs with recall < 1.0: {len(low_recall_ids)}")
    print(f"IDs: {low_recall_ids}")
    print(f"Average Recall across all {len(all_recalls)} queries: {sum(all_recalls)/len(all_recalls):.4f}")

In [ ]:
index.describe()

In [ ]:
#For deleting (int filter) - lt
int_rate_delete = "99p"  # change to 1p, 50p, 80p, 99p

# Map int_rate to lower bound (gte value)
range_map = {
    "1p":  10000,
    "50p": 500000,
    "80p": 800000,
    "99p": 990000,
}
gte_value = range_map[int_rate_delete]

result = index.delete_with_filter([{"id": {"$lt": gte_value}}])
print(f"Deleted vectors with id < {gte_value}")
print(result)

In [ ]:
index.describe()

In [ ]:
#For reinserting deleted vectors (int filter)
import pyarrow.parquet as pq
import os
import time

# User inputs
int_rate_insert = "99p"  # change to 1p, 50p, 80p, 99p
batch_size = 1000

# Map int_rate to range
range_map = {
    "1p":  [0, 9999],
    "50p": [0, 499999],
    "80p": [0, 799999],
    "99p": [0, 989999],
}
filter_range = range_map[int_rate_insert]
low, high = filter_range

# Build full paths
dataset_path = os.path.join(base_path, dataset_folder)
emb_path = os.path.join(dataset_path, "shuffle_train.parquet")

# Process shuffle_train in batches to avoid RAM issues
emb_file = pq.ParquetFile(emb_path)
total_inserted = 0

start = time.perf_counter()
for batch in emb_file.iter_batches(batch_size=batch_size):
    batch_df = batch.to_pandas()

    # Filter by id range
    batch_df = batch_df[(batch_df["id"] >= low) & (batch_df["id"] <= high)]

    if batch_df.empty:
        continue

    batch_vectors = []
    for _, row in batch_df.iterrows():
        vector_data = {
            "id": str(row["id"]),
            "vector": row["emb"],
            "meta": {"id": row["id"]},
            "filter": {
                "id": row["id"],
            }
        }
        batch_vectors.append(vector_data)

    index.upsert(batch_vectors)
    total_inserted += len(batch_vectors)
    print(f"Upserted {len(batch_vectors)} vectors, total so far: {total_inserted}")
end = time.perf_counter()
print(f"Done! Total inserted: {total_inserted} vectors with id range {filter_range}")
print("Time taken:", end-start)

In [ ]:
index.describe()